# 🎬 Bollywood Box Office Predictor
## Notebook 4: Feature Engineering

**Goal:**
- Load all data from Notebooks 1, 2, 3
- Merge financial, engagement, sentiment and sarcasm features
- Analyze correlations and validate hypotheses
- Handle missing values
- Build final feature matrix for ML model
- Save clean feature set

### Features Built
```
💰 Financial     → budget, ROI label
🎬 Content       → genre, language, sequel, remake novelty
⭐ Star Power    → cast score, director score
🎵 Music         → views, consistency, reach, sentiment
📹 Trailer       → views, engagement, sentiment
💬 Sentiment     → VADER, TextBlob, Transformer, adjusted
🎭 Custom Scores → theatre, OTT, controversy, hype
🤣 Sarcasm       → emoji, pattern, transformer detection
```

In [ ]:
# ── CELL 2: INSTALL & IMPORTS ─────────────────────────────────────────────
!pip install scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Libraries imported')

In [ ]:
# ── CELL 3: MOUNT DRIVE & LOAD DATA ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH    = '/content/drive/MyDrive/Bollywood_Predictor/'

movies_df    = pd.read_csv(f'{SAVE_PATH}movies_enriched.csv')
sentiment_df = pd.read_csv(f'{SAVE_PATH}sentiment_scores.csv')

print(f'✅ Movies enriched  : {movies_df.shape}')
print(f'✅ Sentiment scores : {sentiment_df.shape}')
print(f'\n📊 Label distribution:')
print(movies_df['label'].value_counts())
print(f'\n📊 Movies loaded:')
for m in movies_df['name'].tolist():
    print(f'   • {m}')

In [ ]:
# ── CELL 4: MERGE ALL FEATURES ────────────────────────────────────────────

# Rename sentiment movie column to match
sentiment_df = sentiment_df.rename(columns={'movie': 'name'})

# Merge movies + sentiment
df = movies_df.merge(sentiment_df, on='name', how='left')

print(f'✅ Merged shape: {df.shape}')
print(f'\n📊 All available columns after merge:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:03d}. {col}')

In [ ]:
# ── CELL 5: HANDLE MISSING VALUES ────────────────────────────────────────

print('📊 Missing values before handling:')
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print('   No missing values!')

# Fill numeric nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col]    = df[col].fillna(median_val)
        print(f'   Filled {col} with median: {median_val:.4f}')

# Fill categorical nulls
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')

print(f'\n✅ Missing values after handling: {df.isnull().sum().sum()}')

In [ ]:
# ── CELL 6: FINAL FEATURE SELECTION ──────────────────────────────────────

feature_cols = [
    # ── Financial
    'budget_cr',

    # ── Content flags
    'is_sequel',
    'is_remake',
    'is_controversy',

    # ── Star power
    'cast_power_score',
    'director_score',
    'combined_star_score',

    # ── Release context
    'release_timing_score',
    'language_score',

    # ── Genre (one-hot)
    'genre_Action',
    'genre_Comedy',
    'genre_Drama',
    'genre_Mythology',
    'genre_RomCom',

    # ── Sequel legacy
    'part1_collection',
    'part1_verdict_score',
    'franchise_momentum',
    'franchise_gap_score',
    'beat_sequel_expectation',
    'sequel_expectation_cr',
    'same_cast',

    # ── Remake features
    'remake_novelty',
    'remake_risk_score',
    'original_verdict_score',

    # ── Trailer engagement
    'trailer_views',
    'trailer_likes',
    'trailer_like_ratio',
    'trailer_engagement',

    # ── Music engagement
    'music_views_1',
    'music_views_2',
    'total_music_views',
    'music_consistency',
    'top_song_reach_score',
    'music_both_strong',
    'music_like_ratio',
    'music_to_trailer_ratio',
    'total_youtube_views',

    # ── Overall sentiment (sarcasm-adjusted)
    'adjusted_mean_score',
    'adjusted_pos_ratio',
    'adjusted_neg_ratio',
    'vader_mean_compound',
    'tb_mean_polarity',
    'tb_mean_subjectivity',

    # ── Trailer specific sentiment
    'trailer_sentiment_score',
    'trailer_positive_ratio',
    'trailer_negative_ratio',
    'trailer_sarcasm_rate',
    'trailer_theatre_score',
    'trailer_ott_score',
    'trailer_controversy',
    'trailer_hype',

    # ── Music specific sentiment
    'music_sentiment_score',
    'music_positive_ratio',
    'music_negative_ratio',
    'music_sarcasm_rate',
    'music_hype',

    # ── Trailer vs Music gap
    'trailer_music_sentiment_gap',

    # ── Sarcasm
    'sarcasm_rate',
    'emoji_sarcasm_rate',
    'pattern_sarcasm_rate',

    # ── Custom keyword scores
    'theatre_excitement_score',
    'ott_preference_score',
    'theatre_vs_ott_ratio',
    'controversy_intensity',
    'hype_density',

    # ── Volume
    'total_comments',
    'trailer_comment_count',
    'music_comment_count',
]

# Remove duplicates and filter to existing columns only
feature_cols = list(dict.fromkeys(feature_cols))
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df['label'].copy()

print(f'✅ Feature matrix ready')
print(f'   Features found : {X.shape[1]}')
print(f'   Samples        : {X.shape[0]}')

# Show which features were not found
all_requested = [
    'budget_cr','is_sequel','is_remake','is_controversy',
    'cast_power_score','director_score','combined_star_score',
    'release_timing_score','language_score',
    'genre_Action','genre_Comedy','genre_Drama','genre_Mythology','genre_RomCom',
    'part1_collection','part1_verdict_score','franchise_momentum',
    'franchise_gap_score','beat_sequel_expectation','sequel_expectation_cr','same_cast',
    'remake_novelty','remake_risk_score','original_verdict_score',
    'trailer_views','trailer_likes','trailer_like_ratio','trailer_engagement',
    'music_views_1','music_views_2','total_music_views','music_consistency',
    'top_song_reach_score','music_both_strong','music_like_ratio',
    'music_to_trailer_ratio','total_youtube_views',
    'adjusted_mean_score','adjusted_pos_ratio','adjusted_neg_ratio',
    'vader_mean_compound','tb_mean_polarity','tb_mean_subjectivity',
    'trailer_sentiment_score','trailer_positive_ratio','trailer_negative_ratio',
    'trailer_sarcasm_rate','trailer_theatre_score','trailer_ott_score',
    'trailer_controversy','trailer_hype',
    'music_sentiment_score','music_positive_ratio','music_negative_ratio',
    'music_sarcasm_rate','music_hype','trailer_music_sentiment_gap',
    'sarcasm_rate','emoji_sarcasm_rate','pattern_sarcasm_rate',
    'theatre_excitement_score','ott_preference_score','theatre_vs_ott_ratio',
    'controversy_intensity','hype_density',
    'total_comments','trailer_comment_count','music_comment_count',
]
missing_features = [c for c in all_requested if c not in df.columns]
if missing_features:
    print(f'\n⚠️  Features not found (skipped):')
    for m in missing_features:
        print(f'   • {m}')
else:
    print(f'\n✅ All requested features found!')

print(f'\n📊 Label distribution:')
print(y.value_counts())

In [ ]:
# ── CELL 7: CORRELATION ANALYSIS ─────────────────────────────────────────

X_corr         = X.copy()
X_corr['roi_pct'] = df['roi_pct']

corr_matrix = X_corr.corr()
roi_corr    = corr_matrix['roi_pct'].drop('roi_pct').sort_values(ascending=False)

print('📊 Top 15 features positively correlated with ROI:')
print(roi_corr.head(15).round(3).to_string())
print('\n📊 Top 5 features negatively correlated with ROI:')
print(roi_corr.tail(5).round(3).to_string())

# Validate key hypotheses
print('\n🔍 Hypothesis Validation:')
print('─' * 50)

hypotheses = [
    ('total_music_views',           '🎵 Music virality'),
    ('trailer_music_sentiment_gap', '🎭 Trailer-Music sentiment gap'),
    ('franchise_momentum',          '🎬 Franchise momentum'),
    ('remake_novelty',              '🎭 Remake novelty'),
    ('theatre_excitement_score',    '🎪 Theatre excitement'),
    ('trailer_sentiment_score',     '📹 Trailer sentiment'),
]

for col, label in hypotheses:
    if col in X_corr.columns:
        corr = X_corr[[col,'roi_pct']].corr().iloc[0,1]
        strength = 'Strong ✅' if abs(corr) > 0.5 else 'Moderate ⚠️' if abs(corr) > 0.3 else 'Weak ❌'
        print(f'   {label:<30}: {corr:>7.3f}  {strength}')

# Heatmap of top 15 correlated features
top_features = roi_corr.abs().nlargest(15).index.tolist() + ['roi_pct']
top_corr     = X_corr[top_features].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(
    top_corr, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    linewidths=0.5, square=True,
    annot_kws={'size': 7}
)
plt.title('Top 15 Features — Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 8: FEATURE IMPORTANCE ────────────────────────────────────────────

le     = LabelEncoder()
y_enc  = le.fit_transform(y)

rf_quick = RandomForestClassifier(
    n_estimators = 200,
    max_depth    = 4,
    random_state = 42,
    class_weight = 'balanced'
)
rf_quick.fit(X, y_enc)

importance_df = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': rf_quick.feature_importances_
}).sort_values('importance', ascending=False)

print('📊 Top 20 Most Important Features:')
print(importance_df.head(20).to_string(index=False))

# Category importance breakdown
categories = {
    'Financial'    : ['budget_cr', 'sequel_expectation_cr'],
    'Star Power'   : ['cast_power_score', 'director_score', 'combined_star_score'],
    'Sequel'       : ['franchise_momentum', 'part1_verdict_score', 'franchise_gap_score', 'beat_sequel_expectation'],
    'Remake'       : ['remake_novelty', 'remake_risk_score'],
    'Music'        : ['total_music_views', 'music_consistency', 'top_song_reach_score', 'music_both_strong'],
    'Trailer'      : ['trailer_views', 'trailer_like_ratio', 'trailer_engagement'],
    'Sentiment'    : ['adjusted_mean_score', 'trailer_sentiment_score', 'music_sentiment_score', 'trailer_music_sentiment_gap'],
    'Custom Scores': ['theatre_excitement_score', 'ott_preference_score', 'controversy_intensity', 'hype_density'],
    'Sarcasm'      : ['sarcasm_rate', 'emoji_sarcasm_rate'],
}

print('\n📊 Feature Importance by Category:')
print('─' * 45)
for category, cols in categories.items():
    existing = [c for c in cols if c in feature_cols]
    if existing:
        cat_importance = importance_df[
            importance_df['feature'].isin(existing)
        ]['importance'].sum()
        print(f'   {category:<15} : {cat_importance:.4f} ({cat_importance*100:.1f}%)')

# Plot top 20
plt.figure(figsize=(12, 9))
top20  = importance_df.head(20)
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top20)))
plt.barh(top20['feature'][::-1], top20['importance'][::-1],
         color=colors[::-1], alpha=0.85)
plt.title('Top 20 Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 9: SAVE FINAL FEATURE MATRIX ────────────────────────────────────

final_df                  = X.copy()
final_df['name']          = df['name'].values
final_df['label']         = df['label'].values
final_df['label_num']     = le.transform(df['label'])
final_df['roi_pct']       = df['roi_pct'].values
final_df['budget_cr']     = df['budget_cr'].values
final_df['collection_cr'] = df['collection_cr'].values

final_df.to_csv(f'{SAVE_PATH}final_features.csv', index=False)

print(f'✅ Final feature matrix saved')
print(f'   Shape   : {final_df.shape}')
print(f'   Features: {len(feature_cols)}')
print(f'\n📊 Feature categories summary:')
print(f'   Financial      : budget, sequel expectation')
print(f'   Content        : genre, language, sequel, remake')
print(f'   Star Power     : cast score, director score')
print(f'   Music          : views, consistency, reach, sentiment')
print(f'   Trailer        : views, engagement, sentiment')
print(f'   Sentiment      : VADER, TextBlob, Transformer, adjusted')
print(f'   Custom Scores  : theatre, OTT, controversy, hype')
print(f'   Sarcasm        : emoji, pattern, transformer detection')
print(f'\n✅ Notebook 4 complete — proceed to Notebook 5: Prediction Model')